# Week 5 - Representation and Dimensionality Report

This notebook builds the Week 5 feature matrices, runs PCA/SVD, and saves the artifacts used in the report.

## 1) Setup

Define paths, import libraries, and validate that Week 3 processed data is available.

In [1]:
from pathlib import Path
import json
import pyarrow
import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px
from scipy import sparse
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent
if not (project_root / "data").exists():
    project_root = project_root.parent

DATA_DIR = project_root / "data" / "processed" / "week03_v1"
PROCESSED_DIR = project_root / "data" / "processed" / "week05"
ARTIFACTS_DIR = project_root / "artifacts" / "week05"
RANDOM_STATE = 42

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    "movies_catalog.parquet",
    "movie_genres.parquet",
    "ratings_clean.parquet",
    "tags_clean.parquet",
]

missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Missing Week 3 processed files: {missing_files}")

DATA_DIR

WindowsPath('c:/Users/jayka/Documents/Projects/big-data-tf/data/processed/week03_v1')

## 2) Load Week 3 Processed Tables

In [2]:
movies_df = pl.read_parquet(DATA_DIR / "movies_catalog.parquet")
movie_genres_df = pl.read_parquet(DATA_DIR / "movie_genres.parquet")
ratings_df = pl.read_parquet(DATA_DIR / "ratings_clean.parquet")
tags_df = pl.read_parquet(DATA_DIR / "tags_clean.parquet")

pl.DataFrame([
    {"table": "movies", "rows": movies_df.height, "cols": movies_df.width},
    {"table": "movie_genres", "rows": movie_genres_df.height, "cols": movie_genres_df.width},
    {"table": "ratings", "rows": ratings_df.height, "cols": ratings_df.width},
    {"table": "tags", "rows": tags_df.height, "cols": tags_df.width},
])

table,rows,cols
str,i64,i64
"""movies""",62423,9
"""movie_genres""",107245,2
"""ratings""",25000095,5
"""tags""",1093351,6


## 3) Build Feature Matrices

In [4]:
def build_genre_features(movies_df: pl.DataFrame, movie_genres_df: pl.DataFrame) -> pl.DataFrame:
    if "genre" in movie_genres_df.columns:
        base = movie_genres_df.select(["movieId", "genre"])
    elif "genres" in movie_genres_df.columns:
        base = (
            movie_genres_df.select(["movieId", "genres"])
            .with_columns(
                pl.col("genres").fill_null("").str.split("|").alias("genre")
            )
            .explode("genre")
            .filter(pl.col("genre") != "")
            .select(["movieId", "genre"])
)
    elif "genres" in movies_df.columns:
        base = (
            movies_df.select(["movieId", "genres"])
            .with_columns(
                pl.col("genres").fill_null("").str.split("|").alias("genre")
            )
            .explode("genre")
            .filter(pl.col("genre") != "")
            .select(["movieId", "genre"])
)
    else:
        return movies_df.select("movieId")

    return base.to_dummies(columns=["genre"]).group_by("movieId").sum()


def build_rating_features(ratings_df: pl.DataFrame) -> pl.DataFrame:
    if "timestamp" in ratings_df.columns:
        ratings_df = ratings_df.with_columns(pl.col("timestamp").cast(pl.Int64))

    agg = ratings_df.group_by("movieId").agg(
        [
            pl.count().alias("rating_count"),
            pl.col("rating").mean().alias("rating_mean"),
            pl.col("rating").std().alias("rating_std"),
            pl.col("rating").min().alias("rating_min"),
            pl.col("rating").max().alias("rating_max"),
            pl.col("rating").median().alias("rating_median"),
            pl.col("timestamp").min().alias("rating_first_ts"),
            pl.col("timestamp").max().alias("rating_last_ts"),
        ]
)

    if "rating_first_ts" in agg.columns and "rating_last_ts" in agg.columns:
        agg = agg.with_columns(
            ((pl.col("rating_last_ts") - pl.col("rating_first_ts")) / 86400.0).alias(
                "rating_span_days"
            )
        )

    return agg


def build_tag_text(tags_df: pl.DataFrame) -> pl.DataFrame:
    tag_col = None
    for candidate in ["tag", "tags", "tag_text"]:
        if candidate in tags_df.columns:
            tag_col = candidate
            break

    if tag_col is None:
        return pl.DataFrame({"movieId": [], "tag_text": [], "tag_count": []})

    return (
        tags_df.select(["movieId", tag_col])
        .with_columns(pl.col(tag_col).cast(pl.Utf8))
        .group_by("movieId")
        .agg([
            pl.col(tag_col).unique().alias("tag_list"),
            pl.count().alias("tag_count"),
        ])
        .with_columns(pl.col("tag_list").list.join(" ").alias("tag_text"))
        .select(["movieId", "tag_text", "tag_count"])
)


def build_text_corpus(movies_df: pl.DataFrame, tags_agg: pl.DataFrame) -> pl.DataFrame:
    text_df = movies_df.select(["movieId", "title"]).with_columns(
        pl.col("title").cast(pl.Utf8).fill_null("")
    )

    if tags_agg.height > 0:
        text_df = text_df.join(tags_agg, on="movieId", how="left")
    else:
        text_df = text_df.with_columns(pl.lit("").alias("tag_text"))

    text_df = text_df.with_columns(
        (
            pl.col("title").fill_null("")
            + pl.lit(" ")
            + pl.col("tag_text").fill_null("")
        ).alias("text_blob")
    )

    return text_df.select(["movieId", "text_blob", "tag_count"])


def build_feature_matrices(
    movies_df: pl.DataFrame,
    movie_genres_df: pl.DataFrame,
    ratings_df: pl.DataFrame,
    tags_df: pl.DataFrame,
    max_features: int = 5000,
    ) -> tuple[pd.DataFrame, sparse.csr_matrix, dict]:
    genre_features = build_genre_features(movies_df, movie_genres_df)
    rating_features = build_rating_features(ratings_df)
    tag_agg = build_tag_text(tags_df)

    numeric_features = rating_features.join(genre_features, on="movieId", how="left")
    if "tag_count" in tag_agg.columns:
        numeric_features = numeric_features.join(
            tag_agg.select(["movieId", "tag_count"]), on="movieId", how="left"
        )

    numeric_features = numeric_features.fill_null(0).sort("movieId")
    text_df = build_text_corpus(movies_df, tag_agg).sort("movieId")
    corpus = text_df.get_column("text_blob").to_list()

    vectorizer = TfidfVectorizer(
        max_features=max_features, min_df=2, stop_words="english"
    )
    tfidf_matrix = vectorizer.fit_transform(corpus)

    return numeric_features.to_pandas(), tfidf_matrix, vectorizer.vocabulary_


numeric_df, tfidf_matrix, vocab = build_feature_matrices(
    movies_df, movie_genres_df, ratings_df, tags_df
 )

pl.from_pandas(numeric_df).write_parquet(PROCESSED_DIR / "movie_numeric_features.parquet")
sparse.save_npz(str(PROCESSED_DIR / "movie_text_tfidf.npz"), tfidf_matrix)
vocab_serializable = {key: int(value) for key, value in vocab.items()}
(PROCESSED_DIR / "movie_text_tfidf_vocab.json").write_text(
    json.dumps(vocab_serializable, indent=2, ensure_ascii=True)
 )

numeric_df.head()

C:\Users\jayka\AppData\Local\Temp\ipykernel_14752\3601873363.py:36: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("rating_count"),
C:\Users\jayka\AppData\Local\Temp\ipykernel_14752\3601873363.py:73: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("tag_count"),


,movieId,rating_count,rating_mean,rating_std,rating_min,rating_max,rating_median,rating_first_ts,rating_last_ts,rating_span_days,...,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western,tag_count
0,1,57309,3.893708,0.921552,0.5,5.0,4.0,822873600,1574285022,8696.891458,...,0,0,0,0,0,0,0,0,0,697
1,2,24228,3.251527,0.959851,0.5,5.0,3.0,822873600,1574276821,8696.796539,...,0,0,0,0,0,0,0,0,0,180
2,3,11804,3.142028,1.008443,0.5,5.0,3.0,823185228,1573439445,8683.497882,...,0,0,0,0,1,0,0,0,0,29
3,4,2523,2.853547,1.108531,0.5,5.0,3.0,823185225,1574213055,8692.451736,...,0,0,0,0,1,0,0,0,0,11
4,5,11714,3.058434,0.996611,0.5,5.0,3.0,823185224,1573033018,8678.793912,...,0,0,0,0,0,0,0,0,0,24


## 4) Dimensionality Reduction and Comparison

In [5]:
def run_pca(numeric_df: pd.DataFrame, random_state: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    movie_ids = numeric_df["movieId"].values
    numeric_only = numeric_df.drop(columns=["movieId"])
    scaler = StandardScaler()
    numeric_scaled = scaler.fit_transform(numeric_only)

    pca = PCA(n_components=0.9, random_state=random_state)
    pca_result = pca.fit_transform(numeric_scaled)

    pca_df = pd.DataFrame(pca_result)
    pca_df.insert(0, "movieId", movie_ids)

    variance_df = pd.DataFrame({
        "component": np.arange(1, len(pca.explained_variance_ratio_) + 1),
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_variance": np.cumsum(pca.explained_variance_ratio_),
    })

    return pca_df, variance_df


def run_svd(tfidf_matrix: sparse.csr_matrix, random_state: int) -> tuple[np.ndarray, pd.DataFrame]:
    n_components = min(200, tfidf_matrix.shape[1] - 1) if tfidf_matrix.shape[1] > 1 else 1
    svd = TruncatedSVD(n_components=n_components, random_state=random_state)
    svd_result = svd.fit_transform(tfidf_matrix)

    variance_df = pd.DataFrame({
        "component": np.arange(1, len(svd.explained_variance_ratio_) + 1),
        "explained_variance_ratio": svd.explained_variance_ratio_,
        "cumulative_variance": np.cumsum(svd.explained_variance_ratio_),
    })

    return svd_result, variance_df


def reconstruction_error_pca(numeric_df: pd.DataFrame, random_state: int) -> float:
    numeric_only = numeric_df.drop(columns=["movieId"])
    scaler = StandardScaler()
    numeric_scaled = scaler.fit_transform(numeric_only)

    pca = PCA(n_components=0.9, random_state=random_state)
    transformed = pca.fit_transform(numeric_scaled)
    reconstructed = pca.inverse_transform(transformed)
    return float(np.mean((numeric_scaled - reconstructed) ** 2))


def reconstruction_error_svd(tfidf_matrix: sparse.csr_matrix, random_state: int) -> float:
    n_components = min(200, tfidf_matrix.shape[1] - 1) if tfidf_matrix.shape[1] > 1 else 1
    svd = TruncatedSVD(n_components=n_components, random_state=random_state)
    transformed = svd.fit_transform(tfidf_matrix)
    reconstructed = svd.inverse_transform(transformed)
    diff = tfidf_matrix - sparse.csr_matrix(reconstructed)
    return float((diff.multiply(diff)).sum() / tfidf_matrix.shape[0])


def run_tsne(embedding: np.ndarray, random_state: int) -> np.ndarray:
    tsne = TSNE(n_components=2, random_state=random_state, init="random", perplexity=30)
    return tsne.fit_transform(embedding)


pca_df, pca_variance = run_pca(numeric_df, RANDOM_STATE)
svd_embeddings, svd_variance = run_svd(tfidf_matrix, RANDOM_STATE)

pl.from_pandas(pca_df).write_parquet(PROCESSED_DIR / "movie_pca_embeddings.parquet")
svd_columns = [f"svd_{idx + 1}" for idx in range(svd_embeddings.shape[1])]
pl.DataFrame(svd_embeddings, schema=svd_columns).write_parquet(
    PROCESSED_DIR / "movie_svd_embeddings.parquet"
 )

pca_variance.to_csv(ARTIFACTS_DIR / "pca_variance.csv", index=False)
svd_variance.to_csv(ARTIFACTS_DIR / "svd_variance.csv", index=False)

pca_error = reconstruction_error_pca(numeric_df, RANDOM_STATE)
svd_error = reconstruction_error_svd(tfidf_matrix, RANDOM_STATE)

comparison_table = pd.DataFrame([
    {
        "method": "PCA",
        "components": int(pca_variance["component"].max()),
        "cumulative_variance": float(pca_variance["cumulative_variance"].max()),
        "reconstruction_error": pca_error,
    },
    {
        "method": "TruncatedSVD",
        "components": int(svd_variance["component"].max()),
        "cumulative_variance": float(svd_variance["cumulative_variance"].max()),
        "reconstruction_error": svd_error,
    },
])
comparison_table.to_csv(ARTIFACTS_DIR / "comparison_table.csv", index=False)

tsne_input = pca_df.drop(columns=["movieId"]).values
tsne_coords = run_tsne(tsne_input, RANDOM_STATE)
tsne_df = pd.DataFrame(tsne_coords, columns=["tsne_1", "tsne_2"])
tsne_df.insert(0, "movieId", pca_df["movieId"].values)
tsne_df.to_parquet(ARTIFACTS_DIR / "pca_tsne.parquet", index=False)

comparison_table

,method,components,cumulative_variance,reconstruction_error
0,PCA,20,0.901221,0.098779
1,TruncatedSVD,200,0.357191,0.637321


## 5) Visualization Set

In [7]:
def save_plot(fig, path: Path) -> None:
    fig.write_html(str(path))
    png_path = path.with_suffix(".png")
    try:
        fig.write_image(str(png_path))
    except Exception:
        pass

pca_variance = pd.read_csv(ARTIFACTS_DIR / "pca_variance.csv")
svd_variance = pd.read_csv(ARTIFACTS_DIR / "svd_variance.csv")
tsne_df = pd.read_parquet(ARTIFACTS_DIR / "pca_tsne.parquet")

fig_pca = px.line(
    pca_variance,
    x="component",
    y="cumulative_variance",
    markers=True,
    title="PCA Cumulative Explained Variance (Numeric Features)",
 )
save_plot(fig_pca, ARTIFACTS_DIR / "pca_cumulative_variance.html")
fig_pca.show()

fig_svd = px.line(
    svd_variance,
    x="component",
    y="cumulative_variance",
    markers=True,
    title="SVD Cumulative Explained Variance (Text TF-IDF)",
 )
save_plot(fig_svd, ARTIFACTS_DIR / "svd_cumulative_variance.html")
fig_svd.show()

fig_tsne = px.scatter(
    tsne_df,
    x="tsne_1",
    y="tsne_2",
    title="t-SNE on PCA Embeddings",
    opacity=0.6,
 )
save_plot(fig_tsne, ARTIFACTS_DIR / "tsne_scatter.html")
fig_tsne.show()